# CompSci 590.7 Assignment 4: Belief States and Partial Observability


Name: **[ENTER HERE]**

Welcome to Assignment 4. This notebook will help you understand:
- Partial observability.
- Belief states.
- Updating belief states.
- RNNs for solving partially observable environments.


## Preliminaries
You'll need three imports to complete this assigment:
- numpy: The fundamental package for scientific computing with Python.
- matplotlib: Used for scientific visualization of data in Python.
- gymnasium: A ubiquitous library for standard environments for reinforcement learning.
```console
pip install gymnasium
```
- gridworld: A module containing the 4x3 GridWorld we've been working with.
- pygame: A module required for visualizaing the cartpole environment.
```console
pip install pygame
```
- PyTorch: A neural network library. Install by following their [installation page](https://pytorch.org/get-started/locally/), be sure to install the CPU version.
- tools: A module for helper functions and plotting.

Please do not change the imports and answer the written questions in a separate document. If you run into issues installing the libraries please let us know ASAP. 

In [ ]:
%%capture
%matplotlib inline

In [ ]:
import copy
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from pomdp import MDP, TigerPOMDP, TMaze
from tools import functional_feature_value_determination, eval_rnn_q, discounted_return, plot_greedy_policy, plot_tiger_value_function
from q_learning import q_learning

## Section 1: Partial Observability and Belief States

In this assignment we use the Tiger POMDP environment discussed in class.

<img src="tigerpomdp.webp" alt="tigerpomdp" width="400"/>

It was first introduced as a POMDP in *POMDP: Introduction to Partially Observable Markov Decision Processes* with a quote:

> A tiger is put with equal probability behind one of two doors, while treasure is put behind the other one. You are standing in front of the two closed doors and need to decide which one to open. If you open the door with the tiger, you will get hurt (negative reward). But if you open the door with treasure, you receive a positive reward. Instead of opening a door right away, you also have the option to wait and listen for tiger noises. But waiting is neither free nor entirely accurate. You might hear the tiger behind the left door while it is actually behind the right door and vice versa.

We model this environment as follows. We provide the indices in the brackets, wherever necessary. You have three actions: `right` (0), `left` (1), `wait` (2). `right` and `left` choose the right and left doors, and `wait` is used to obtain a noisy observation. The agent can be in one of three states: tiger is behind the right door (0), tiger is behind the left door (1) or the terminal state. The agent recieves one of three observations: 

[1, 0, 0] -> Roar heard from right, `tiger-right`,

[0, 1, 0] -> Roar heard from left, `tiger-left`,

[0, 0, 1] -> Terminal observation, `terminal`.

The agent's initial state is chosen equally likely from the first two states (0, 1) and therefore the initial state distribution is:
$$d_0 = [1/2, 1/2, 0.0]^{\intercal}.$$

Upon waiting, the agent hears the roar from the door behind which the tiger is with probability 0.85 and the other door with probabiltiy 0.15. You can see how the agent behaves for a policy that randomly choses a door after waiting 5 times.

In [ ]:
pomdp = TigerPOMDP()

observation, _ = pomdp.reset()
terminal = False
observations = []
num_steps = 0
while num_steps < 5:
    observation, r, terminal, _, _ = pomdp.step(2) # wait
    observations.append(observation)
    num_steps += 1
choose_door = 0 if np.random.uniform() < 0.5 else 1
observation, r, _, _, _ = pomdp.step(choose_door)
observations.append(observation)
print(f"Door Choice: {choose_door}")
if r < 0.0:
    print("💀💀💀")
else:
    print("💰💰💰")
observations_str = [pomdp.describe_observation(observation) for observation in observations]
" -> ".join(observations_str)

### Belief State Updates

Here we work with the assumption that the agent is somehow "aware" of the underlying set of states $\mathcal S$. It would be helpful for the agent to be able to maintain a set of beliefs regarding which state it is at. Therefore, a belief state $b$ is a probability distribution over $\mathcal S$ and $b(s)$ denotes assigned to world state $s$ by belief state $b$. As per the axioms of probability $ 0 \leq b(s) \leq 1$ and $\sum_{s \in \mathcal S} b(s) = 1$.

Consider the Tiger POMDP above. Initially, the belief state is $[0.5, 0.5, 0.0]$. If the agent takes action `wait` and also hears the roar from the right belief state becomes $[0.85, 0.15, 0.0]$. Suppose it takes action `wait` again and once again hears the roar from the right, then the probability concentrates in state 0: $[0.969, 0.0302, 0.0]$. Intuitively, the agent has higher confidence of the tiger being behind the right door.

We further need to be able to update the agent's beliefs based on the observations. The state estimator must compute a new beliefe state $b'$ given an old belief state $b$, an action $a$, and an observation $o$. The new degree of belief is some state $s'$, $b'(s')$, can be obtained as follows:

$$b'(s') = \frac{\Pr(o | s', a) \sum_{s \in \mathcal S} \Pr(s'| a, b, s) \Pr(s | a, b)}{\Pr(o | a, b)},$$

where $\Pr(o | a, b)$ is a normalizing term that esnures that the belief state probabilities, $b'(s')$ sum to 1. We want you to implement this belief update rule below. You can find the derivation for this belief update rule in Section 19.2 of the Algorithms for Decision Making book. Please keep your code general enough for it to run with a general POMDP object.

In [ ]:
def belief_update(pomdp: TigerPOMDP, b: np.ndarray, o: np.ndarray, a: int) -> np.ndarray:
    """
    Given the POMDP, current belief state, an observation o, and the action a you return the 
    new belief state. You have to use the fuction observation_probability to obtain Pr(). Notice Pr(o | a, b) 
    in the denomiator is simply a normalizing term. You can use this fact to simplify your 
    calculation of the new belief state.
    """
    b_new = np.zeros_like(b)
    b_new[0] = 1.0 #This is a place holder
    
    ### START CODE HERE ###
    
    
    
    ### END CODE HERE ###
    
    return b_new

Now that we have a mechanism to estimate belief states we can define a "belief MDP". It is defined as follows:

- $\mathcal B$ is the set of belief states, comprise the state space
- $\mathcal A$ is the set of actions which remains the same as the POMDP
- The transitions are deterministic and defined by the belief_update function above
- The new reward function $R'(b, a) = \sum_{s \in \mathcal S} b(s) R(s, a)$

## QUESTION
1.1 Your initial belief state is $b_0 = \begin{bmatrix} 0.5 \\ 0.5 \end{bmatrix}$. You take a `wait` action and observe $z = \texttt{tiger-left}$. What is your new belief state $b_1$ after this observation? Show all steps of your work.

We programatically define this belief MDP below in the TigerBeliefMDP class.

In [ ]:
class TigerBeliefMDP(MDP):
    """
    Here we provide you an implementation of the belief MDP for the simple POMDP above. 
    """
    
    def __init__(self, gamma: float = 0.5):
        """
        You initialise this with a discount factor and the underlying POMDP is instantiated
        in the init function. Notice how the start state is the start distribution of the 
        underlying Tiger POMDP.
        """
        self.pomdp = TigerPOMDP(gamma=gamma)
        super().__init__(self.pomdp.T, self.pomdp.R, self.pomdp.p0, gamma=self.pomdp.gamma)
    
    @property
    def features_size(self) -> int:
        return self.pomdp.p0.shape[0]
    
    def reset(self) -> tuple[np.ndarray, dict]:
        observation, _ = self.pomdp.reset()
        self.current_state = self.pomdp.p0
        return self.current_state, {'observation': observation}
    
    def featurized_reset(self) -> tuple[np.ndarray, dict]:
        observation, flags_dict = self.reset()
        return self.quadratic_features(observation), flags_dict
    
    
    def featurized_step(self, action: int) -> tuple[np.ndarray, float, bool, bool, dict]:
        observation, reward, terminal, truncation, flags_dict = self.step(action)
        return self.quadratic_features(observation), reward, terminal, truncation, flags_dict
    
    
    def step(self, action: int) -> tuple[np.ndarray, float, bool, bool, dict]:
        """
        Given an action make a call to the step funciton of the underlying POMDP.
        Finally, we call the belief_update function to perform one step transitions
        in the belief space.
        """
        observation, reward, terminal, truncation, variables_dict = self.pomdp.step(action)
        
        # Call to belief_update you implemented above
        self.current_state = belief_update(self.pomdp, self.current_state, observation, action)
        
        variables_dict["observation"] = observation #Observation from the underlying POMDP
        np.testing.assert_almost_equal(self.current_state.sum(axis=-1), 1.0, decimal=3, 
                                    err_msg="Belief state does not sum to 1 " + str(self.current_state))
        
        return self.current_state, reward, terminal, truncation, variables_dict


Now we can perform rollout in the belief MDP as one would in any MDP except now we return observations, beliefs, actions, and rewards. You can run this rollout code to verify that your implementation matches the example given above. 

In [ ]:
def rollout(env: TigerBeliefMDP, pi, max_steps=np.inf) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    We take as input an environment and policy pi. We perform a single rollout.
    This rollout takes an action according to the policy pi and transitions according
    to the step function.
    """
    # Four lists for observations, beliefs, rewards and features
    ep_observations = []
    ep_beliefs = []
    ep_actions = []
    ep_rewards = []

    beliefs, variables_dict = env.reset()
    #ep_observations.append(variables_dict["observation"])
    ep_beliefs.append(beliefs)
    
    terminal, truncation = False, False
    num_steps = 0

    while (not terminal) and num_steps <= max_steps:
        if num_steps == max_steps:
            action = 0
        else:
            action = pi(beliefs)
        beliefs, reward, terminal, truncation, variables_dict = env.step(action)
        ep_observations.append(variables_dict["observation"])
        ep_beliefs.append(beliefs)
        ep_actions.append(action)
        ep_rewards.append(reward)
        num_steps += 1

    return np.array(ep_observations), np.array(ep_beliefs), np.array(ep_actions), np.array(ep_rewards)


In [ ]:
belief_mdp = TigerBeliefMDP()
def wait(any_input):
    return 2
ep_observations, ep_beliefs, ep_actions, ep_rewards =  rollout(belief_mdp, wait, max_steps=5)
ep_observations, ep_beliefs, ep_actions, ep_rewards

Now that we're more familiar with belief updates, let's go more in depth into how POMDP planning works and alpha vectors. Below we have a POMDP plan for the Tiger problem. Each POMDP plan has an action associated with it (as its root), in this case the `wait` action is at the root of this plan. After taking the `wait` action, we have two possible observations: `tiger-left` and `tiger-right`. Depending on which observation we've seen, we're going to pick the opposite door: we pick `right` for the `tiger-left` observation, and `left` for the `tiger-right` observation. The plan is visualized below.

<img src="tiger_plan.png" alt="Tiger Plan" width="300"/>

Since we know the environment dynamics in POMDP planning (the transition matrix, reward matrix and observation function), we can calculate the two-step returns (or expected utility) of this plan given that we start at each state. The vector over states of expected utilities following this plan for each state is our alpha vector for this plan. Since belief states are a distribution over all states you could be in in the current time step, the dot product of the two gives the expected utility of this belief state.

We're going to calculate the alpha vector for this two-step plan. Here's how:
1. Start out with picking a state.
2. Our plan is associated with a root action. For this given action, there are a set of possible observations you could see, and a probability associated with emitting this observation (and a reward too, but `wait`ing has 0 reward), given an action and a state. For each of these observations, we need to calculate the return on the second step, discount it, and weight by this probability of seeing this observation.
3. We now calculate the return on our second step. Conditional on seeing this observation, we see a second state (which doesn't change in the case of Tiger, since the tiger always stays in the same door) our plan chooses a second action. We get a reward by taking this action, which gives our second-step return.
4. Multiply by the corresponding probabilities and sum all possible branches together, to get an element in your alpha vector corresponding to the original state you conditioned on.

## QUESTION
1.2 You have the following plan: `wait` first, then if you hear `tiger-left`, open `right`. If you hear `tiger-right`, open `left`. Derive the two-step policy alpha vector for this plan.

Thanks to our knowledge of the underlying dynamics we are now ready to evaluate a near optimal policy in this simple belief MDP. We will do this using online TD. The difference, from previous assignments (Assignment 2), is that we estimate the  value function, $V(s)$, in a tabular manner as opposed to a linear function approximation. We would like to achieve an approximation of the convex "bowl" structure of the state-action value functions with the probability $\Pr(s = 0)$ on the x-axis, as discussed in class.

In [ ]:
def online_td(policy, env: TigerBeliefMDP, alpha: float=0.01, num_samples=1000):
    """
    Performs online TD for the tabular setting.
    Learns a value function by updating the value 
    by using the 1-step bootstrapped estimate as the target.
    If you need a reference, check SB page 203 or Assignment 2.
    The marked difference is that you are estimating values for individual 
    p = Pr(s = 0). Where p is the probability the agent is state 0 
    (Tiger Right) or quite simply belief_state[0]. We expect you to return a
    dictionary mapping from the probability p to the value under the policy.
    See the rollout function above to understand how to simulate the belief
    MDP.
    
    policy: is the policy for which we estimate the q values
    env: is the environment
    alpha: the step size or learning rate
    num_samples: is the number of samples
    """
    
    samples_so_far = 0
    values_dict = {0.5: 0.0, 0.85: 0.0, 0.15: 0.0} #Populate this dict where
    #each element in the dictionary maps from Pr(s = 0)
    #to the value estiamte of the policy passed as an argument.
    #Further, values_dict[p] = value of acting under policy 
    #with belief Pr(s = 0) = p. The set of observations are all 
    #the values that are observed using the policy.
    
    discounted_returns_dict = [{}, {}, {}]
    
    while samples_so_far <= num_samples:
        prev_beliefs, variables_dict = env.reset()
        
        terminal = False
        while not terminal:
            action = policy(prev_beliefs)
            beliefs, reward, terminal, truncation, variables_dict = env.step(action)
            samples_so_far += 1
            discount = env.gamma if not terminal else 0
            belief_value_estimate = 0.0 #Initial value estimate
            if beliefs[0] in values_dict:
                belief_value_estimate = values_dict[beliefs[0]]
            target = reward + discount*belief_value_estimate
            
            prev_belief_value_estimate = 0.0
            if prev_beliefs[0] in values_dict:
                prev_belief_value_estimate = values_dict[prev_beliefs[0]]
            
            values_dict[prev_beliefs[0]] = prev_belief_value_estimate + alpha*(target - prev_belief_value_estimate)
            
            samples_so_far += 1
            prev_beliefs = beliefs
                
    return values_dict


In [ ]:
def simple_threshold_policy(belief_state):
    action = 2 #LISTEN
    if belief_state[0] <= 0.031:
        #Belief that Tiger is very likely NOT to RIGHT
        action = 0 # Go Right
    elif belief_state[1] <= 0.031:
        #Belief that Tiger is very likely NOT to LEFT
        action = 1 # Go Left
    return action

belief_mdp = TigerBeliefMDP()
values_dict = online_td(simple_threshold_policy, belief_mdp, alpha=0.0001, num_samples=int(2e6))
values_dict

Plot the value function learned above. If you were to pick the best possible action, meaning the value function that is the greatest, you should obtain a convex "bowl" like structure. This bowl is also piece-wise linear.

In [ ]:
plot_tiger_value_function(values_dict)

<!-- Thanks to our knowledge of the underlying dynamics we are now ready to learn an optimal policy in this Simple belief MDP. We will use Q-learning with linear function approximation. -->
Now we'll consider how to solve POMDPs using *only samples*. We'll do that with recurrent neural networks in the next section.

## Section 2: Solving Partially Observable Environments with RNNs

How can we solve these partially observable problems without access to the underlying dynamics? The answer is the same as how we do sequence modelling in deep learning: with RNNs! In order to test our RNN, we'll be using another classic partially observable environment, inspired by memory testing in neuroscience called Bakker's T-Maze.

### Bakker's T-Maze
<img src="tmaze.png" alt="T-Maze" width="500"/>

The T-Maze is a memory-testing environment, where the agent first observes either a red or blue observation in its initial observation. Afterwards, it has to traverse down a hallway (where the observations are all the same hallway observation), and finally reaches a junction. If the agent saw the blue initial observation, the positive rewarding (+4) goal state is to the `north` of the junction, whereas going `south` from the junction will result in a negative reward (-0.1) terminal state. If the agent initially saw the red observation, the converse is true. The main task of the agent is to remember its initial observation when it's at the junction, so it knows which direction to go.

In this part of the assignment, you'll be implementing a recurrent Q-learning agent, with a gated recurrent unit (GRU) as the recurrent network. Let's first implement the RNN Q-function. In previous assignments, you implemented Q-functions as a linear function between state features and weights, $Q(\phi(s_t), a_t) = \mathbf{w}_a^{\intercal}\phi(s_t)$. Now we'll be using a *recurrent neural network* as our Q-function. If you need a good reference to how recurrent neural networks work, we refer you to page 370 of the [deep learning book](https://www.deeplearningbook.org/contents/rnn.html). Let $Q_{\theta} : \mathcal{O} \times \mathcal{A} \times \mathcal{H} \rightarrow \mathbb{R}^{|A|} \times \mathcal{H}$ represent our recurrent Q-function neural network, parameterized by $\theta$. Our Q-function takes as input the previous action taken, the current observation seen, and the previous hidden state as input. Here, $\mathcal{H}$ represents the space of our hidden states (normally $\mathbb{R}^d$, where $d$ is the dimension of your hidden state), which is meant to be a condensed version of our history of observations and actions. Our Q-function takes as input the observation $\mathbf{o}_t$, the previous action $a_{t - 1}$ and previous hidden state $\mathbf{h}_{t - 1}$ and outputs the Q-values of _all_ actions at time step $t$, as well as the next hidden state $\mathbf{h}_t$.

We'll be using the PyTorch library to implement our neural network. If you need a reference to implementing RNNs, check out this tutorial on [RNN classification with PyTorch](https://pytorch.org/tutorials/intermediate/char_rnn_classification_tutorial.html). If you need a reference to how GRUs work, check out the [PyTorch documentation for GRUs](https://pytorch.org/docs/stable/generated/torch.nn.GRU.html).

In [ ]:
class RecurrentQ(nn.Module):
    def __init__(self, input_size: int, n_actions: int, hidden_size: int = 16):
        """
        The recurrent Q-function. Input size here should be the size of the observation vector,
        plus the size of a one-hot encoded action.
        """
        super(RecurrentQ, self).__init__()
        self.hidden_size = hidden_size
        self.input_size = input_size
        self.n_actions = n_actions

        self.gru = nn.GRU(input_size, self.hidden_size)
        self.value = nn.Linear(self.hidden_size, n_actions)
        
    def forward(self, obs: torch.tensor, hs: torch.tensor):
        """
        Forward pass of our network.
        obs: observation, of size (t x b x obs_size).
        hs: initial hidden state, of size (1 x b x obs_size). 
            The leading 1 dimension is n_layers = 1.
        This forward function will apply this recurrent function across all
        t timesteps in the observations.
        
        Returns q-values of shape (t x b x n_actions) as well as the 
        final hidden states, of size (b x hidden_size)
        """
        qs = torch.zeros((obs.shape[0], obs.shape[1], self.n_actions))
        final_hs = torch.zeros_like(hs)
        
        ### START CODE HERE ###
        
        
        
        ### END CODE HERE ###
        
        return qs, final_hs

    def init_hidden_state(self, batch_size: int):
        """
        Initialize the GRU hidden state, of size (n_layers x batch_size x hidden_size).
        We always use n_layers = 1.
        """
        return torch.zeros((1, batch_size, self.hidden_size))

Let's initialize our environment and pass in a single time step to see what we get. We should end up with q-values of shape (t x batch_size x n_actions) (or (1, 1, 2)) and next hidden states of size (1 x batch_size x hidden_size) (or (1, 1, 16)).

NOTE: our Q-function here takes as input the most recent observation $o_t$ concatenated with the one-hot encoded _previous_ action $a_{t - 1}$ as input. This allows us to condition on histories of observation-action pairs, as opposed to only observations. This is an engineering "trick" that helps the RNN reduce partial observability. You might notice that we don't have a "previous action" before our initial observation. We simply use the zero vector (of size `n_action`) as a placeholder for this vector.

In [ ]:
def one_hot_action(a: int, n_actions: int):
    return np.eye(n_actions)[a]
    
def add_dims_and_convert(arr: np.array):
    return torch.tensor(arr, dtype=torch.float).unsqueeze(0).unsqueeze(0)

env = TMaze()

q_func = RecurrentQ(env.observation_space.n + env.action_space.n, env.action_space.n, hidden_size=16)
hs = q_func.init_hidden_state(1)

obs, _ = env.reset()
obs_prev_action = np.concatenate((obs, np.zeros(env.action_space.n)))

# Add time and batch dimensions to our observation
obs_prev_action = add_dims_and_convert(obs_prev_action)

qs, next_hs = q_func(obs_prev_action, hs)
qs.shape, next_hs.shape

Now let's define our Q-learning function!

In [ ]:
def eps_greedy(q_values: np.ndarray, epsilon: float = 0.1):
    """
    Sample an epsilon-greedy action. Given a set of Q-values, this
    function will sample the greedy (w.r.t. the Q-values) action
    w.p. 1 - epsilon, and with probability epsilon, will select
    a random action from ALL actions.
    NOTE: q_values should be of shape (n_actions), and NOT
    (1 x n_actions).
    """

    index = torch.argmax(q_values).item()
    pi = np.zeros(q_values.shape[-1])
    pi[index] = (1 - epsilon)
    pi += epsilon / pi.shape[0]
    chosen_action = np.random.choice(pi.shape[0], p=pi)

    return chosen_action


def recurrent_q_learning(q: nn.Module, env: gym.Env,
                         n_episodes: int = int(1e3),
                         epsilon: float = 0.1,
                         max_episode_length: int = 200,
                         learning_rate: float = 1e-3):
    """
    Recurrent Q-learning with the Adam optimizer.
    Since we have to do backpropagation through time, we need to collect an entire
    trajectory (episode) before doing an update.
    """

    optimizer = torch.optim.Adam(q.parameters(), lr=learning_rate)

    step = 0
    returns = []
    losses = []
    lengths = []

    for eps in range(n_episodes):
        optimizer.zero_grad()
        qs = None

        """
        Here, you should:
        1. initialize the RNN latent state, of size (1 x 1 x hidden_size).
        2. Reset your environment, receive an initial observation of size (obs_size).
        3. Concatenate an array of zeros, of size (n_actions), to your initial observation, and
           Use add_dims_and_convert to add a batch dim and convert to a torch.tensor of shape (1 x (obs_size + n_actions)).
        4. calculate your q-values as well as the next latent state, of sizes (1 x n_actions) and (1 x 1 x hidden_size)
        5. select your first action, epsilon greedily.
        Remember to make use of the helper functions add_dims_and_convert and eps_greedy here.
        Also take note of the shape of input to eps_greedy.
        """
        ### START CODE HERE ###

        

        ### END CODE HERE ###

        # Arrays to collect experience from this episode
        eps_qs, eps_all_next_qs, eps_actions, eps_rewards, eps_discounts = [qs], [], [], [], []

        terminal, truncated = False, False

        while not (terminal or truncated) and (eps_steps < max_episode_length):
            """
            In this block:
            1. take a step in the environment
            2. calculate your next q-values (remember to concatenate the previous one-hot action to the observation!), 
               as well as next latent state
            3. get your next epsilon-greedy action, based on these q-values.
            Remember to make use of the helper function one_hot_action!
            """
            reward, next_action, next_qs = None, None, None

            ### START CODE HERE ###

            

            ### END CODE HERE ###

            eps_steps += 1
            step += 1

            discount = env.gamma if not terminal else 0

            eps_actions.append(action)
            eps_rewards.append(reward)
            eps_discounts.append(discount)
            eps_qs.append(next_qs)

            action = next_action

        # Convert our episode data into torch.tensor's to calculate our
        # q-learning loss.
        eps_qs = torch.concat(eps_qs, dim=0)
        eps_actions = torch.tensor(eps_actions).unsqueeze(-1)
        eps_rewards = torch.tensor(eps_rewards).unsqueeze(-1)
        eps_discounts = torch.tensor(eps_discounts).unsqueeze(-1)

        # Get all our Q(o_t, a_t)'s here.
        # We need to do some fancy indexing here to get the Q-values for the
        # corresponding actions we took.
        all_qs = eps_qs[:-1]
        t_indices, b_indices = torch.meshgrid(torch.arange(eps_actions.shape[0]), torch.arange(eps_actions.shape[1]),
                                              indexing='ij')
        q_a = all_qs[t_indices, b_indices, eps_actions]

        """
        In this final block:
        1. calculate your q-learning targets. REMEMBER to use the .detach() function on these targets to
           prevent gradients from flowing through your estimated targets for our semi-gradient update.
        2. calculate your mean-squared error loss between q_a and the target.
        3. call .backward() on your calculated loss, and update your neural network with the optimizer.
        """

        ### START CODE HERE ###

        

        ### END CODE HERE ###

        returns.append(discounted_return(eps_rewards, env.gamma).item())
        losses.append(loss.detach().item())
        lengths.append(eps_steps)

        if eps > 0 and eps % 200 == 0:
            print(f"Episode {eps}. Avg length {sum(lengths) / len(lengths):.2f}. "
                  f"Avg loss: {sum(losses) / len(losses):.3f}, "
                  f"avg discounted returns: {sum(returns) / len(returns):.3f}")
            losses = []
            returns = []
            lengths = []

    return q

Now let's train our recurrent Q-function on our simple POMDP.

In [ ]:
q_func = RecurrentQ(env.observation_space.n + env.action_space.n, env.action_space.n, hidden_size=16)

trained_q = recurrent_q_learning(q_func, env, n_episodes=int(5e3), epsilon=0.1, learning_rate=5e-4)

To get a baseline for our performance, let's initialize a new RNN and evaluate on that first.

In [ ]:
random_q_func = RecurrentQ(env.observation_space.n + env.action_space.n, env.action_space.n, hidden_size=16)

random_episode_rewards, _ = eval_rnn_q(random_q_func, env, n_episodes=100, max_episode_steps=100)
all_random_returns = np.array([discounted_return(rs, env.gamma) for rs in random_episode_rewards])
all_random_returns.mean()

Now let's see how our trained Q-function performs.

In [ ]:
episode_rewards, _ = eval_rnn_q(trained_q, env, n_episodes=100, max_episode_steps=100)
all_returns = np.array([discounted_return(rs, env.gamma) for rs in episode_rewards])
all_returns.mean()

### Wrapping up

Congratulations! Please submit your assignment to the autograder. Remember to use the following command to export the python code.

```console
jupyter nbconvert --to python --RegexRemovePreprocessor.patterns="^%" CPS590_7_assignment4.ipynb
```

We have tried to build up your intuition for partial observability, as well as sample-based ways to solve partial observability in this assignment. We hope it's been a good learning experience!